# Notebook 3 — Synthetic Nigerian Reviews: Cross-Domain Cultural Augmentation

Generates **~1,000 culturally-aware synthetic Nigerian reviews** spread evenly across all
NaijaReview domains, using LangChain + Gemini.

**Design decisions:**
- Each domain gets `~100 reviews` (9 domains + general = ~1,000 total)
- Source reviews are sampled from `integrated_final_dataset.jsonl` for that domain
- Each prompt encodes **region-specific cultural cues** (Lagos, Abuja, PH, Kano, etc.)
- **Sentiment is preserved** exactly from the original review — no hallucinated ratings
- Output is appended to `integrated_final_dataset.jsonl` with `source: synthetic_naija_v2`
- Rate limiting: 2s sleep between calls; batch processing with checkpoint resume

> Run Notebook 2 first to ensure all domains are extracted.

In [ ]:
# ── Cell 1: Mount Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
# Only installs what isn't already present in Colab
!pip install -q langchain-core langchain-google-genai

In [ ]:
# ── Cell 3: Config ───────────────────────────────────────────────────────────
import os
from google.colab import userdata

BASE       = "/content/drive/MyDrive/NaijaReview_Data"
PROC_DIR   = f"{BASE}/Naiview/data/processed"
SOURCE_FILE = f"{PROC_DIR}/integrated_final_dataset.jsonl"   # input
OUT_FILE    = f"{PROC_DIR}/integrated_final_dataset.jsonl"   # APPEND
CKPT_FILE   = f"{PROC_DIR}/.synth_v2_checkpoint.json"        # resume support

# ── Synthesis targets per domain (must sum to ~1000) ────────────────────────
DOMAIN_TARGETS = {
    'food':          130,
    'health':        100,
    'hospitality':   100,
    'retail':        100,
    'services':       90,
    'automotive':    100,
    'personal_care': 100,
    'entertainment': 100,
    'education':      80,
    'general':        50,
}
TOTAL_TARGET = sum(DOMAIN_TARGETS.values())
print(f"Total synthetic reviews to generate: {TOTAL_TARGET}")
print("Domain targets:", DOMAIN_TARGETS)

# Configure Gemini
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
print("API key configured.")

In [ ]:
# ── Cell 4: Cultural context bank ────────────────────────────────────────────
# These are the cultural cues injected per domain to make reviews authentic.
# Regions, vernacular references, and Nigerian-specific concerns per domain.

DOMAIN_CULTURAL_CONTEXT = {
    'food': {
        'regions':    ['Lagos Island', 'Victoria Island', 'Surulere', 'Lekki',
                       'Wuse Abuja', 'Yaba', 'Port Harcourt GRA', 'Ikeja'],
        'concerns':   ['value for money', 'generator / light situation', 'portion size',
                       'if food is local or continental', 'pepper and spice level',
                       'whether you can eat and pay or must pay first'],
        'expressions': ['This food dey sweet die!', 'The pepper was too much abeg.',
                        'Portion small like this?', 'Worth the price for VI',
                        'This place is a banger!', 'I no fit come back here.',
                        'Mama put vibes but premium pricing'],
    },
    'health': {
        'regions':    ['Lagos Island General Hospital', 'LASUTH', 'UCH Ibadan',
                       'National Hospital Abuja', 'LUTH', 'St. Nicholas Lagos'],
        'concerns':   ['waiting time / long queue', 'doctor availability',
                       'cost of consultation', 'NHIS / HMO acceptance',
                       'cleanliness of the facility', 'drug availability',
                       'whether NEPA (power) was stable during consultation'],
        'expressions': ['The queue was insane, spent half the day waiting.',
                        'Doctor was thorough sha.', 'Pharmacy no get the drug.',
                        'They accept HMO which is a big plus.',
                        'Clean environment, rare for naija hospitals.',
                        'Staff attitude needs serious improvement.'],
    },
    'hospitality': {
        'regions':    ['Victoria Island Lagos', 'Maitama Abuja', 'Asokoro',
                       'Port Harcourt', 'Ikeja GRA', 'Lekki Phase 1'],
        'concerns':   ['constant light / generator supply', 'WiFi strength',
                       'breakfast inclusion', 'security / gate situation',
                       'room size vs price', 'staff friendliness',
                       'proximity to the airport or business district'],
        'expressions': ['Light was stable throughout, 10/10.',
                        'WiFi was a joke — they need to fix that.',
                        'Room was neat and spacious for the price.',
                        'Security was top notch, felt safe.',
                        'Breakfast was fire — jollof and eggs every morning!',
                        'Location no good for someone without a car.'],
    },
    'retail': {
        'regions':    ['Balogun Market Lagos', 'Computer Village Ikeja',
                       'Wuse Market Abuja', 'Tejuosho Market', 'Ikeja Mall',
                       'The Palms Lekki', 'Shoprite'],
        'concerns':   ['price vs quality', 'authenticity of goods',
                       'parking situation', 'customer service attitude',
                       'availability of POS / card payment',
                       'whether they do returns', 'delivery option'],
        'expressions': ['Price too high abeg, Balogun dey give am cheaper.',
                        'Quality is good sha, worth it.',
                        'No POS? In this economy?',
                        'They delivered same day, big surprise!',
                        'Customer service attitude was rude.',
                        'Nice arrangement, feels like abroad.'],
    },
    'services': {
        'regions':    ['Ikeja', 'Surulere', 'Garki Abuja', 'Ojota', 'Festac',
                       'Trans Amadi PH', 'Gwarinpa Abuja'],
        'concerns':   ['punctuality / keeping to time', 'whether they give receipt',
                       'professionalism', 'whether they inflate the bill',
                       'ability to reach them on phone', 'quality of work done'],
        'expressions': ['They came on time, which is rare in naija.',
                        'Did a clean job, no complaints.',
                        'They inflated the bill at the end, very annoying.',
                        'Phone calls go unanswered after job is done.',
                        'Professional and efficient — recommended!',
                        'They tried to add extra charges for no reason.'],
    },
    'automotive': {
        'regions':    ['Ladipo Market Lagos', 'Nnewi auto spare parts',
                       'Berger Lagos', 'Wuse auto mechanics Abuja',
                       'Trans Amadi Port Harcourt', 'Challenge Ibadan'],
        'concerns':   ['whether they inflate repair costs', 'availability of parts',
                       'how long the job takes', 'whether work is guaranteed',
                       'trust issues with mechanic', 'use of original parts vs Tokunbo'],
        'expressions': ['This mechanic is honest, which is a miracle.',
                        'They used fake parts, had to go back in two weeks.',
                        'Work done well and at fair price — rare!',
                        'They quoted 50k and finished at 90k. Classic.',
                        'Quick diagnosis, knew the problem straight away.',
                        'Original parts only, no Tokunbo rubbish.'],
    },
    'personal_care': {
        'regions':    ['Lekki', 'VI', 'Surulere', 'Festac', 'Agege',
                       'Wuse Abuja', 'Old GRA Port Harcourt'],
        'concerns':   ['price vs quality of the style', 'how long the wait was',
                       'whether they burn hair or skin', 'cleanliness of equipment',
                       'durability of the style', 'whether they rush the job'],
        'expressions': ['Hair came out correct! She's talented.',
                        'Burnt my ear with the dryer, never again.',
                        'Took 4 hours for a simple cut — no time management.',
                        'Clean salon, good vibes, great service.',
                        'Price was high but the result was worth it.',
                        'My nails lasted two full weeks, she's a pro.'],
    },
    'entertainment': {
        'regions':    ['Lagos Island', 'Lekki-Ajah', 'Victoria Island',
                       'Wuse 2 Abuja', 'Ikeja', 'GRA Port Harcourt'],
        'concerns':   ['crowd size and safety', 'music selection',
                       'drink prices vs outside bars', 'parking and exit situation',
                       'AC / ventilation in the venue', 'if they check ID'],
        'expressions': ['Vibes were on 100, DJ was mad!',
                        'Drink price is robbery for what you get.',
                        'Crowd was too rowdy, not safe.',
                        'Perfect spot for a relaxed Friday night.',
                        'AC no dey work, we were sweating throughout.',
                        'Security was tight, felt comfortable going with the girls.'],
    },
    'education': {
        'regions':    ['Yaba Lagos', 'Asokoro Abuja', 'GRA Ibadan',
                       'New Haven Enugu', 'UNILAG area', 'Ahmadu Bello way Kaduna'],
        'concerns':   ['quality of instructors', 'class size',
                       'curriculum relevance to naija job market',
                       'certification recognition', 'cost vs value',
                       'online vs physical class options'],
        'expressions': ['The instructor was knowledgeable and patient.',
                        'Class was overcrowded, hard to concentrate.',
                        'Certificate is not recognized anywhere, waste of money.',
                        'Practical training was excellent, got a job after.',
                        'Online option was convenient for my schedule.',
                        'Affordable fee for the quality of teaching.'],
    },
    'general': {
        'regions':    ['Lagos', 'Abuja', 'Port Harcourt', 'Ibadan', 'Kano', 'Enugu'],
        'concerns':   ['value for money', 'customer service', 'overall experience'],
        'expressions': ['Honestly worth trying.', 'No be bad place.',
                        'I go recommend for anybody.', 'E get as e be.',
                        'Average experience, nothing to shout about.',
                        'Exceeded my expectations.'],
    },
}

print("Cultural context bank loaded for domains:", list(DOMAIN_CULTURAL_CONTEXT.keys()))

In [ ]:
# ── Cell 5: Load source reviews from integrated dataset ──────────────────────
# We sample from the real Yelp reviews in the integrated dataset per domain.
# Only naija_mode=False records (real reviews) are used as source material.

import json
import random
from collections import defaultdict

random.seed(42)

# Pool of real reviews to rewrite, keyed by domain
source_pool = defaultdict(list)

with open(SOURCE_FILE, 'r') as f:
    for line in f:
        try:
            rec = json.loads(line)
            domain = rec.get('domain', 'general')
            # Only use original Yelp records, not already synthetic ones
            if rec.get('source', '').startswith('synthetic'):
                continue
            if len(rec.get('text', '')) < 50:
                continue
            source_pool[domain].append(rec)
        except:
            pass

print("Source review pool (real records only):")
for d in sorted(source_pool.keys()):
    print(f"  {d:<22} {len(source_pool[d]):>6,} reviews")

In [ ]:
# ── Cell 6: LangChain + Gemini setup ────────────────────────────────────────
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.75,   # some variability for natural text
    max_output_tokens=512,
)

# ── Prompt template ──────────────────────────────────────────────────────────
# Encoding Nigerian cultural nuance as a first-class constraint, not an afterthought.
PROMPT_TEMPLATE = PromptTemplate.from_template("""\
You are a Nigerian consumer writing an online review. Rewrite the review below so it sounds
exactly like a real Nigerian wrote it — not a foreigner imitating Nigerians.

DOMAIN: {domain}
REVIEWER REGION: {region}
ORIGINAL SENTIMENT: {sentiment} ({stars}★)

CULTURAL RULES (follow all of them):
1. Use Nigerian English naturally — not every sentence needs Pidgin, but the voice must feel Nigerian.
2. Inject 1–3 of these domain-specific cultural concerns where relevant: {concerns}
3. You may use one of these Nigerian expressions if it fits naturally: {expressions}
4. Reference the Nigerian context: mention things like generator/light situation, POS, NEPA,
   Nigerian food items, area reputation, naira pricing, or local equivalents where appropriate.
5. The star rating ({stars}★) MUST be preserved. Do not change the sentiment.
6. Keep it realistic — 2 to 5 sentences. Not every review needs all Pidgin.
7. Do NOT start with 'I' as the first word. Vary the opening.
8. Return ONLY the review text. No intro, no quotes, no labels.

ORIGINAL REVIEW:
{original_text}
""")

chain = PROMPT_TEMPLATE | llm
print("LangChain chain configured.")

In [ ]:
# ── Cell 7: Load checkpoint (for resume if interrupted) ──────────────────────
import time

# Track which review_ids we've already synthesised (avoids duplication on rerun)
completed_source_ids = set()
synthetic_count_by_domain = defaultdict(int)

if os.path.exists(CKPT_FILE):
    with open(CKPT_FILE, 'r') as f:
        ckpt = json.load(f)
        completed_source_ids = set(ckpt.get('completed_source_ids', []))
        synthetic_count_by_domain = defaultdict(int, ckpt.get('counts', {}))
    print(f"Resuming from checkpoint: {len(completed_source_ids)} source reviews already processed.")
    print("Synthetic counts so far:", dict(synthetic_count_by_domain))
else:
    print("No checkpoint found — starting fresh.")

def save_checkpoint():
    with open(CKPT_FILE, 'w') as f:
        json.dump({
            'completed_source_ids': list(completed_source_ids),
            'counts': dict(synthetic_count_by_domain),
        }, f)

print("Checkpoint system ready.")

In [ ]:
# ── Cell 8: Main synthesis loop ──────────────────────────────────────────────
# Generates synthetic Nigerian reviews domain by domain.
# Checkpoints every 25 records. Sleeps 2s per API call to avoid rate limits.

import uuid
import random

total_generated = sum(synthetic_count_by_domain.values())
errors = 0

with open(OUT_FILE, 'a') as out_f:

    for domain, target in DOMAIN_TARGETS.items():

        already_done  = synthetic_count_by_domain.get(domain, 0)
        still_needed  = target - already_done

        if still_needed <= 0:
            print(f"[{domain}] Already generated {already_done}/{target} — skipping.")
            continue

        pool = [r for r in source_pool.get(domain, [])
                if r.get('review_id', '') not in completed_source_ids]

        if not pool:
            print(f"[{domain}] Source pool exhausted or empty — skipping.")
            continue

        # Sample without replacement
        sample = random.sample(pool, min(still_needed, len(pool)))
        ctx    = DOMAIN_CULTURAL_CONTEXT.get(domain, DOMAIN_CULTURAL_CONTEXT['general'])

        print(f"\n[{domain}] Generating {len(sample)} reviews "
              f"(have {already_done}, need {target})...")

        for i, source_rec in enumerate(sample):

            region      = random.choice(ctx['regions'])
            concerns    = ', '.join(random.sample(ctx['concerns'],
                                                  min(3, len(ctx['concerns']))))
            expressions = ' | '.join(random.sample(ctx['expressions'],
                                                    min(3, len(ctx['expressions']))))

            try:
                response = chain.invoke({
                    'domain':        domain,
                    'region':        region,
                    'sentiment':     source_rec.get('sentiment', 'neutral'),
                    'stars':         source_rec.get('stars', 3),
                    'concerns':      concerns,
                    'expressions':   expressions,
                    'original_text': source_rec['text'][:600],  # cap input length
                })
                naija_text = response.content.strip()

                # Basic quality filter
                if len(naija_text) < 40:
                    errors += 1
                    continue

                # Estimate pidgin density (rough heuristic)
                pidgin_markers = ['dey', 'abeg', 'na', 'sha', 'abi', 'oga', 'dem',
                                  'wahala', 'chop', 'naija', 'no be', 'e be like',
                                  'sabi', 'wey', 'comot', 'enter', 'carry']
                words       = naija_text.lower().split()
                pidgin_hits = sum(1 for w in words if w in pidgin_markers)
                pidgin_density = round(min(pidgin_hits / max(len(words), 1), 1.0), 3)

                stars = source_rec.get('stars', 3)
                if stars >= 4:
                    sentiment, tier = 'positive',  ('very_positive' if stars == 5 else 'positive')
                elif stars <= 2:
                    sentiment, tier = 'negative',  ('very_negative' if stars == 1 else 'negative')
                else:
                    sentiment, tier = 'neutral', 'neutral'

                record = {
                    'review_id':              f"synth_v2_{uuid.uuid4().hex[:12]}",
                    'user_id':                source_rec.get('user_id', 'unknown'),
                    'item_id':                source_rec.get('item_id', 'unknown'),
                    'text':                   naija_text,
                    'stars':                  stars,
                    'sentiment':              sentiment,
                    'tier':                   tier,
                    'register':               'pidgin_mixed',
                    'naija_mode':             True,
                    'category':               source_rec.get('category', domain.title()),
                    'yelp_category':          source_rec.get('yelp_category', ''),
                    'domain':                 domain,
                    'category_confidence':    1.0,
                    'region':                 region,
                    'pidgin_density':         pidgin_density,
                    'text_len':               len(naija_text),
                    'quality_score':          0.85,
                    'nigerian_context_score': 0.90,
                    'engagement':             source_rec.get('engagement', 0),
                    'date':                   source_rec.get('date', ''),
                    'source':                 'synthetic_naija_v2',
                    'source_review_id':       source_rec.get('review_id', ''),
                    'eval_text_quality':      True,
                    'eval_rating_accuracy':   True,
                    'eval_cross_domain':      True,
                }

                out_f.write(json.dumps(record) + '\n')
                completed_source_ids.add(source_rec.get('review_id', ''))
                synthetic_count_by_domain[domain] += 1
                total_generated += 1

                # Print progress every 10
                if (i + 1) % 10 == 0:
                    print(f"  [{domain}] {i+1}/{len(sample)} done  "
                          f"(total so far: {total_generated})")

                # Checkpoint every 25
                if total_generated % 25 == 0:
                    out_f.flush()
                    save_checkpoint()

                time.sleep(2)   # Gemini rate limit buffer

            except Exception as e:
                errors += 1
                print(f"  ERROR on review {i}: {e}")
                time.sleep(4)   # longer sleep on error
                continue

        print(f"  [{domain}] Finished. Total for domain: "
              f"{synthetic_count_by_domain[domain]}/{target}")
        save_checkpoint()

print(f"\n{'='*50}")
print(f"Synthesis complete.")
print(f"Total synthetic reviews generated: {total_generated}")
print(f"Errors / skipped: {errors}")

In [ ]:
# ── Cell 9: Quality audit — inspect sample outputs per domain ────────────────
# Loads the last N synthetic records and prints one per domain for manual review.

from collections import defaultdict

samples_by_domain = defaultdict(list)

with open(OUT_FILE, 'r') as f:
    for line in f:
        try:
            rec = json.loads(line)
            if rec.get('source') == 'synthetic_naija_v2':
                samples_by_domain[rec['domain']].append(rec)
        except:
            pass

print("=" * 60)
print("SYNTHETIC REVIEW SAMPLE (1 per domain)")
print("=" * 60)

for domain in sorted(samples_by_domain.keys()):
    records = samples_by_domain[domain]
    sample  = random.choice(records)
    print(f"\n[{domain.upper()}]  ★{sample['stars']}  region={sample['region']}  "
          f"pidgin_density={sample['pidgin_density']}")
    print(f"  {sample['text']}")
    print(f"  ({len(records)} total synthetic records for this domain)")

In [ ]:
# ── Cell 10: Final dataset summary ──────────────────────────────────────────
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

domain_total  = Counter()
domain_synth  = Counter()
domain_real   = Counter()

with open(OUT_FILE, 'r') as f:
    for line in f:
        try:
            rec = json.loads(line)
            d = rec.get('domain', 'unknown')
            domain_total[d] += 1
            if rec.get('source', '').startswith('synthetic'):
                domain_synth[d] += 1
            else:
                domain_real[d] += 1
        except:
            pass

print(f"{'='*60}")
print(f"FINAL DATASET SUMMARY")
print(f"{'='*60}")
print(f"{'Domain':<22} {'Total':>7}  {'Real':>7}  {'Synthetic':>9}  {'Synth%':>7}")
print("-" * 60)

grand_total = sum(domain_total.values())
for d in sorted(domain_total.keys()):
    tot   = domain_total[d]
    real  = domain_real[d]
    synth = domain_synth[d]
    pct   = synth / tot * 100 if tot else 0
    print(f"{d:<22} {tot:>7,}  {real:>7,}  {synth:>9,}  {pct:>6.1f}%")

print("-" * 60)
print(f"{'TOTAL':<22} {grand_total:>7,}")

# Stacked bar chart
domains   = sorted(domain_total.keys())
reals     = [domain_real[d]  for d in domains]
synths    = [domain_synth[d] for d in domains]

x   = np.arange(len(domains))
fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x, reals,  label='Real (Yelp)',     color='#5B8DB8', edgecolor='black')
ax.bar(x, synths, label='Synthetic Naija', color='#E8873A', edgecolor='black',
       bottom=reals)
ax.set_xticks(x)
ax.set_xticklabels(domains, rotation=40, ha='right', fontsize=9)
ax.set_ylabel('Record count')
ax.set_title('Final Dataset: Real vs Synthetic by Domain', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('/content/final_dataset_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("Summary chart saved to /content/final_dataset_summary.png")

In [ ]:
# ── Cell 11: Cleanup checkpoint file (optional — run only if fully done) ─────
# Only run this after you are satisfied with the synthesis output.

# import os
# if os.path.exists(CKPT_FILE):
#     os.remove(CKPT_FILE)
#     print("Checkpoint file removed.")

print("Checkpoint kept for safety. Uncomment the block above to remove it.")
print("\nAll three notebooks are now complete. Your integrated_final_dataset.jsonl")
print("contains real Yelp reviews + domain-expanded records + ~1,000 synthetic Naija reviews.")